### Inital Organization of Forward Modeling
##### Natalie Yurek, Last edit: 05/05/26

##### From vg parameters covert vg parameters -> water content -> electrical conductivity -> predicted TDEM response




Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### Please assign all paramaters/variables/user inputs below!

In [ ]:
#### van Genuchten parameters
alpha = 0.145
n_vg = 2.68
theta_r = 0.045
theta_s = 0.43
m_vg = 1 - 1/n_vg

#### Soil profile parameters
water_table_depth  = 50
profile_depth = 200         # depth of soil profile in cm (100cm = 1 m)
dz = 1                      # layer thickness or depth increment in cm

#### Archie's parameters
sigma_w = 0.1
m_archie = 1.5
n_archie = 2.0

In [ ]:
def validate_van_genuchten_parameters(alpha, n_vg, theta_r, theta_s):
    """ Validate the van Genuchten parameters and throw an error if invalid. """
    if alpha <= 0:
        raise ValueError("Alpha must be greater than 0.")
    if n_vg <= 1:
        raise ValueError("n must be greater than 1.")
    if not (0 <= theta_r < theta_s <= 1):
        raise ValueError("Theta_r must be between 0 and theta_s, and theta_s must be between 0 and 1.")
    return True

In [ ]:
def pressure_head(profile_depth, water_table_depth):
    """ define the pressure head profile for the soil column based on the depth of soil profile and water table depth"""
    z = np.linspace(0, profile_depth, int(profile_depth/dz))  # depth array from 0 to profile_depth
    h = water_table_depth - z  # pressure head is positive above water table and negative below
    return h
h = pressure_head(profile_depth, water_table_depth)

In [ ]:
def wc_to_saturation(theta, theta_s):
    """ Convert volumetric water content to saturation. """
    return theta / theta_s  
sw = wc_to_saturation(theta, theta_s)

## might not neet this if using the simplied Archie's law that only depends on theta?

Water content -> Electrical conductivity

Archies Law 

σ=σ(w)​θ^m

Note: σ_bulk (bulk conductivity) = electrical conductivity of entire soil volume. 
[Which will come from TEM inversion of EC vs Depth profile] 

In [ ]:
def theta_to_conductivity(theta, theta_s, sigma_w, m_archie, n_archie, sw):
    """ convert volumetric water content to bulk electrical conductivity using simplified Archie's law """
    # sigma_bulk = sigma_w * (theta_s ** m_archie) * (sw ** n_archie)
    sigma_bulk = sigma_w * theta**m_archie
    return sigma_bulk

## remove unused functions and variables if only using the simplified Archie's?

In [ ]:
def build_layered_model(z, sigma_bulk, dz):
    """ Convert a continuous conductivity profile into a layered model for forward modeling. """
    layers = []
    for i in range(len(z)-1):
        layer = {
            'top': z[i],
            'bottom': z[i+1],
            'conductivity': sigma_bulk[i]
        }
        layers.append(layer)
    return layers

    ###NAT pls review how this is impportant to the model process before continuing!

###### note: WalkTEM measures voltage vs time from which we infer EC vs time


Everything below this is the old stuff 

Electrical conductivity -> Water content

Archies Law 

σ=σ(w)​θ^m

θ=(σ(w)​σ​)^1/m

Note: σ_bulk (bulk conductivity) = electrical conductivity of entire soil volume. 
Which will come from TEM inversion of EC vs Depth profile

(This COULD come from HYDRUS. It will give water content as a function of depth at a specific time)

In [ ]:
def ec_to_wc_archie(sigma_bulk, sigma_water, m=1.8):
    """
    Convert bulk EC to water content using Archie's law
    sigma_bulk: measured conductivity (S/m)
    sigma_water: pore water conductivity (S/m)
    m: cementation exponent
    """
    theta = (sigma_bulk / sigma_water)**(1/m) ## What water content would produce this bulk conductivity, given how conductive the pore water is?
    
    # clip to physical range
    theta = np.clip(theta, 0.0, 0.6)
    return theta

*Plot to check*

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(theta_profile, z, label="Water Content")
plt.gca().invert_yaxis()
plt.xlabel("Theta (m³/m³)")
plt.ylabel("Depth (m)")
plt.legend()
plt.show()

Water content -> van Genuchten Soil Properties